## Phi-4 RAG with Azure AI Search

This notebook demonstrates how to use Phi-4-mini and Phi-4-multimodal for Retrieval Augmented Generation (RAG) with Azure AI Search. It covers both single-modality (text-only) and multi-modality (text and image) scenarios.

**Prerequisites:**
*   Azure AI Search vector index (follow [these instructions](https://learn.microsoft.com/azure/search/search-get-started-portal-import-vectors?tabs=sample-data-storage%2Cmodel-aoai%2Cconnect-data-storage) to create one)
*   Phi-4-mini OR Phi-4-multimodal endpoints deployed in Microsoft Foundry

> **Note:** The Azure AI Inference beta SDK (`azure-ai-inference`) that earlier versions of this notebook used was [retired on August 26, 2026](https://learn.microsoft.com/azure/foundry-classic/foundry-models/supported-languages). The chat calls now use the [OpenAI SDK](https://github.com/openai/openai-python) with the Microsoft Foundry v1 endpoint ([migration guide](https://learn.microsoft.com/azure/foundry/how-to/model-inference-to-openai-migration)). Set these environment variables:
>
> - `OPENAI_BASE_URL`: `https://<resource>.openai.azure.com/openai/v1/`
> - `OPENAI_API_KEY`: the API key of your Microsoft Foundry resource
> - `MODEL`: the deployment name of your Phi-4-mini model (text-only section)
> - `MULTIMODAL_MODEL`: the deployment name of your Phi-4-multimodal model (multi-modal section)
> - `AZURE_SEARCH_ENDPOINT`, `AZURE_SEARCH_INDEX`, `AZURE_SEARCH_KEY`: your Azure AI Search settings


In [ ]:
! pip install openai azure-search-documents

### Text-Only RAG with Phi-4-mini

This section demonstrates how to use Phi-4-mini as a chat completion model for RAG, using only text as input.  It involves connecting to Microsoft Foundry and Azure AI Search, retrieving relevant documents, and generating an answer using the retrieved context.

In [ ]:
# Step 1: Connect to Microsoft Foundry Inference & Azure AI Search
import os
from openai import OpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

chat_client = OpenAI(
    base_url=os.environ["OPENAI_BASE_URL"], # Microsoft Foundry v1 endpoint
    api_key=os.environ["OPENAI_API_KEY"],
)

search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name=os.environ["AZURE_SEARCH_INDEX"],
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])
)

# Step 2: Retrieve relevant documents from Azure AI Search
def retrieve_documents(query: str, top: int = 10):
    vector_query = VectorizableTextQuery(text=query, k_nearest_neighbors=top, fields="text_vector")
    results = search_client.search(search_text=query,vector_queries=[vector_query],select=["content"],top=top)
    return [doc["content"] for doc in results]

# Step 3: Generate answer using RAG!
def generate_rag_response(query: str):
    docs = retrieve_documents(query)
    context = "\n---\n".join(docs)
    prompt = f"""You are a helpful assistant. Use only the following context to answer the question. If the answer isn't in the context, say 'I don't know'.
    Context: {context} Question: {query} Answer:"""
    response = chat_client.chat.completions.create(
        model=os.environ["MODEL"], # Phi-4-mini deployment name
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

# Example usage
user_query = "What is the capital of France?"
answer = generate_rag_response(user_query)
print(f"Q: {user_query}\nA: {answer}")


### Multi-Modal RAG with Phi-4-multimodal

This section demonstrates how to use Phi-4-multimodal as a chat completion model for RAG, incorporating both text and image input.  It covers connecting to Microsoft Foundry and Azure AI Search, retrieving relevant documents, and generating a multimodal response.

**Note:** You can also perform a multi-vector query if you have both `text_vector` and `image_vector` fields in your Azure AI Search index.

In [ ]:
# Step 1: Connect to Microsoft Foundry & Azure AI Search
import os
from openai import OpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery


chat_client = OpenAI(
    base_url=os.environ["OPENAI_BASE_URL"], # Microsoft Foundry v1 endpoint
    api_key=os.environ["OPENAI_API_KEY"],
)

search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name=os.environ["AZURE_SEARCH_INDEX"],
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])
)

# Step 2: Retrieve relevant documents from Azure AI Search
def retrieve_documents(query: str, top: int = 3):
    vector_query = VectorizableTextQuery(text=query, k_nearest_neighbors=top, fields="text_vector")
    results = search_client.search(search_text=query, vector_queries=[vector_query], select=["content"], top=top)    
    return [doc["content"] for doc in results]

## Example for Muli-modal search if you have a text_vector AND image_vector field in your vector_index
## NOTE, image vectorization is in preview at the time of writing this code, please use azure-search-documents pypi version >11.6.0b6 
# def retrieve_documents_multimodal(query: str, image_url: str, top: int = 3):
#     text_vector_query = VectorizableTextQuery(
#         text=query,
#         k_nearest_neighbors=top,
#         fields="text_vector",
#         weight=0.5  # Adjust weight as needed
#     )
#     image_vector_query = VectorizableImageUrlQuery(
#         url=image_url,
#         k_nearest_neighbors=top,
#         fields="image_vector",
#         weight=0.5  # Adjust weight as needed
#     )

#     results = search_client.search(
#         search_text=query,  
#         vector_queries=[text_vector_query, image_vector_query],
#         select=["content"],
#         top=top
#     )
#     return [doc["content"] for doc in results]


# Step 3: Generate a multimodal RAG-based answer using retrieved text and an image input
def generate_multimodal_rag_response(query: str, image_url: str):
    # Retrieve text context from search
    docs = retrieve_documents(query)
    context = "\n---\n".join(docs)

    # Build a prompt that combines the retrieved context with the user query
    prompt = f"""You are a helpful assistant. Use only the following context to answer the question. If the answer isn't in the context, say 'I don't know'.
    Context: {context} Question: {query} Answer:"""
    # Create a chat request that includes both text and image input
    response = chat_client.chat.completions.create(
        model=os.environ["MULTIMODAL_MODEL"], # Phi-4-multimodal deployment name
        messages=[
            {"role": "system", "content": "You are a helpful assistant that can process both text and images."},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": image_url, "detail": "high"}},
                ],
            },
        ]
    )
    return response.choices[0].message.content

# Example usage
user_query = "Can you search for similar items to this shoe?"
sample_image_url = "https://images.unsplash.com/photo-1542291026-7eec264c27ff?q=80&w=1770&auto=format&fit=crop&ixlib=rb-4.0.3"
answer = generate_multimodal_rag_response(user_query, sample_image_url)
print(f"Q: {user_query}\nA: {answer}")